In [ ]:
import random
import math
from typing import List, Optional, Union
from abc import ABC, abstractmethod

In [ ]:
# Базовый класс для генераторов числовых последовательностей
class SequenceGenerator:
    def __init__(self, start: int = 1):
        self.start = start
        self.current = start

    # Генерирует последовательность чисел
    @abstractmethod
    def generate(self, count: int) -> List[int]:
        pass

    def __iter__(self):
        return self

    # Для использования в циклах
    def __next__(self) -> int:
        result = self._next_number()
        self.current += 1
        return result

    # Генерирует следующее число последовательности
    @abstractmethod
    def _next_number(self) -> int:
        pass

In [ ]:
# Генератор простых чисел
class PrimeGenerator(SequenceGenerator):
    def __init__(self, start: int = 2):
        if start < 2:
            start = 2
        super().__init__(start)
        self.primes_cache = []
        self._fill_cache_until(start)

    # Проверяет, является ли число простым
    def _is_prime(self, n: int) -> bool:
        if n < 2:
            return False
        if n == 2:
            return True
        if n % 2 == 0:
            return False

        # Проверяем делители до корня из n
        limit = int(math.sqrt(n)) + 1
        for i in range(3, limit, 2):
            if n % i == 0:
                return False
        return True

    # Заполняет кэш простых чисел до n
    def _fill_cache_until(self, n: int):
        if not self.primes_cache or self.primes_cache[-1] < n:
            start = self.primes_cache[-1] + 1 if self.primes_cache else 2
            for num in range(start, n + 1):
                if self._is_prime(num):
                    self.primes_cache.append(num)

    # Генерирует следующее простое число
    def _next_number(self) -> int:
        num = self.current
        while True:
            if self._is_prime(num):
                if num not in self.primes_cache:
                    self.primes_cache.append(num)
                return num
            num += 1

    # Генерирует count простых чисел, начиная с текущей позиции
    def generate(self, count: int) -> List[int]:
        result = []
        temp_current = self.current
        for _ in range(count):
            prime = self._next_number()
            result.append(prime)
            self.current = prime + 1
        self.current = temp_current
        return result

In [ ]:
# Генератор чисел Фибоначчи
class FibonacciGenerator(SequenceGenerator):
    def __init__(self, start: int = 0):
        super().__init__(start)
        self.a, self.b = 0, 1
        self._advance_to_start()

    def _advance_to_start(self):
        self.a, self.b = 0, 1
        for _ in range(self.start):
            self.a, self.b = self.b, self.a + self.b

    def _next_number(self) -> int:
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        return result

    def generate(self, count: int) -> List[int]:
        result = []
        temp_a, temp_b = self.a, self.b
        for _ in range(count):
            result.append(self.a)
            self.a, self.b = self.b, self.a + self.b
        self.a, self.b = temp_a, temp_b
        return result

In [ ]:
# Абстрактный базовый класс для геометрических фигур
class Shape(ABC):
    def __init__(self, name: str):
        self.name = name

    # площадь фигуры
    @abstractmethod
    def area(self) -> float:
        pass

    # периметр фигуры
    @abstractmethod
    def perimeter(self) -> float:
        pass

    def __str__(self) -> str:
        return f"{self.name}: площадь={self.area():.2f}, периметр={self.perimeter():.2f}"

class Circle(Shape):
    def __init__(self, radius: float):
        super().__init__("Круг")
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        super().__init__("Прямоугольник")
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

class Triangle(Shape):
    def __init__(self, side1: float, side2: float, side3: float):
        super().__init__("Треугольник")
        self.side1 = side1
        self.side2 = side2
        self.side3 = side3

    def area(self) -> float:
        # Формула Герона
        s = self.perimeter() / 2
        return math.sqrt(s * (s - self.side1) * (s - self.side2) * (s - self.side3))

    def perimeter(self) -> float:
        return self.side1 + self.side2 + self.side3

In [ ]:
# Создание фигур с обработкой ошибок
class ShapeFactory:
    @staticmethod
    def create_shape(shape_type: str, *args) -> Shape:
        try:
            match shape_type.lower():
                case "circle":
                    if len(args) != 1:
                        raise ValueError("Круг требует 1 параметр - радиус")
                    radius = args[0]
                    if radius <= 0:
                        raise ValueError("Радиус должен быть положительным")
                    return Circle(radius)

                case "rectangle":
                    if len(args) != 2:
                        raise ValueError("Прямоугольник требует 2 параметра - ширину и высоту")
                    width, height = args
                    if width <= 0 or height <= 0:
                        raise ValueError("Ширина и высота должны быть положительными")
                    return Rectangle(width, height)

                case "triangle":
                    if len(args) != 3:
                        raise ValueError("Треугольник требует 3 параметра - длины сторон")
                    side1, side2, side3 = args
                    if side1 <= 0 or side2 <= 0 or side3 <= 0:
                        raise ValueError("Все стороны должны быть положительными")
                    # Проверка неравенства треугольника
                    if (side1 + side2 <= side3 or
                        side1 + side3 <= side2 or
                        side2 + side3 <= side1):
                        raise ValueError("Треугольник с такими сторонами не существует")
                    return Triangle(side1, side2, side3)

                case _:
                    raise ValueError(f"Неизвестный тип фигуры: {shape_type}")

        except ValueError as e:
            print(f"Ошибка создания фигуры: {e}")
            raise
        except Exception as e:
            print(f"Неожиданная ошибка: {e}")
            raise

In [ ]:
# Класс калькулятора для работы с фигурами и последовательностями
class GeometryCalculator:
    def __init__(self):
        self.shapes = []
        self.prime_gen = PrimeGenerator()
        self.fib_gen = FibonacciGenerator()

    def add_shape(self, shape: Shape):
        self.shapes.append(shape)

    def process_shape_command(self, command: str, *args):
        try:
            match command:
                case "create":
                    if len(args) < 2:
                        raise ValueError("Недостаточно аргументов. Используйте: create <тип> <параметры...>")
                    shape_type = args[0]
                    shape_args = [float(x) for x in args[1:]]
                    shape = ShapeFactory.create_shape(shape_type, *shape_args)
                    self.add_shape(shape)
                    print(f"Создана фигура: {shape}")
                    return shape

                case "list":
                    if not self.shapes:
                        print("Коллекция фигур пуста")
                    else:
                        print("Коллекция фигур:")
                        for i, shape in enumerate(self.shapes, 1):
                            print(f"  {i}. {shape}")

                case "total_area":
                    total = sum(shape.area() for shape in self.shapes)
                    print(f"Общая площадь всех фигур: {total:.2f}")
                    return total

                case "total_perimeter":
                    total = sum(shape.perimeter() for shape in self.shapes)
                    print(f"Общий периметр всех фигур: {total:.2f}")
                    return total

                case "clear":
                    count = len(self.shapes)
                    self.shapes.clear()
                    print(f"Удалено {count} фигур")

                case _:
                    raise ValueError(f"Неизвестная команда: {command}")

        except ValueError as e:
            print(f"Ошибка выполнения команды: {e}")
        except Exception as e:
            print(f"Неожиданная ошибка: {e}")
        finally:
            print("Команда выполнена!")

    # Обработка команд для работы с последовательностями
    def process_sequence_command(self, command: str, *args):
        try:
            match command:
                case "prime":
                    if not args:
                        count = 10
                    else:
                        count = int(args[0])

                    primes = self.prime_gen.generate(count)
                    print(f"Первые {count} простых чисел: {primes}")
                    return primes

                case "fibonacci":
                    if not args:
                        count = 10
                    else:
                        count = int(args[0])
                    fib_numbers = self.fib_gen.generate(count)
                    print(f"Первые {count} чисел Фибоначчи: {fib_numbers}")
                    return fib_numbers

                case "prime_from":
                    if len(args) < 2:
                        raise ValueError("Используйте: prime_from <начало> <количество>")
                    start = int(args[0])
                    count = int(args[1])
                    temp_gen = PrimeGenerator(start)
                    primes = temp_gen.generate(count)
                    print(f"{count} простых чисел, начиная с {start}: {primes}")
                    return primes

                case _:
                    raise ValueError(f"Неизвестная команда последовательности: {command}")

        except ValueError as e:
            print(f"Ошибка ввода: {e}")
        except Exception as e:
            print(f"Неожиданная ошибка: {e}")
        finally:
            print("Последовательность сгенерирована!")

In [ ]:
# Демонстрация работы всех функций
def demo(self):
        print("ДЕМОНСТРАЦИЯ РАБОТЫ ПРОГРАММЫ")
        print("\n1. РАБОТА С ФИГУРАМИ:")
        commands = [
            ("create", "circle", "5"),
            ("create", "rectangle", "4", "6"),
            ("create", "triangle", "3", "4", "5"),
            ("list",),
            ("total_area",),
            ("total_perimeter",)
        ]

        for command in commands:
            self.process_shape_command(*command)

        print("\n2. ГЕНЕРАТОРЫ ПОСЛЕДОВАТЕЛЬНОСТЕЙ:")

        seq_commands = [
            ("prime", "5"),
            ("fibonacci", "8"),
            ("prime_from", "50", "5")
        ]

        for command in seq_commands:
            self.process_sequence_command(*command)

        print("\n3. ОБРАБОТКА ОШИБОК:")

        error_commands = [
            ("create", "circle", "-1"),  # Отрицательный радиус
            ("create", "triangle", "1", "1", "3"),  # Несуществующий треугольник
            ("create", "unknown", "1"),  # Неизвестная фигура
            ("prime", "invalid"),  # Неверный аргумент
        ]

        for command in error_commands:
            self.process_shape_command(*command) if command[0] == "create" else self.process_sequence_command(*command)

In [14]:
def main():
    calculator = GeometryCalculator()
    print("ГЕОМЕТРИЧЕСКИЙ КАЛЬКУЛЯТОР И ГЕНЕРАТОР ПОСЛЕДОВАТЕЛЬНОСТЕЙ")
    while True:
        print("\nДоступные команды:")
        print("ФИГУРЫ: create <тип> <параметры>, list, total_area, total_perimeter, clear")
        print("ПОСЛЕДОВАТЕЛЬНОСТИ: prime [n], fibonacci [n], prime_from <start> <n>")
        print("ДЕМО: demo, ВЫХОД: exit")

        try:
            user_input = input("\nВведите команду: ").strip()
            if not user_input:
                continue

            if user_input.lower() == 'exit':
                print("До свидания!")
                break
            elif user_input.lower() == 'demo':
                calculator.demo()
                continue

            parts = user_input.split()
            command = parts[0]
            args = parts[1:]

            # Определяем тип команды
            if command in ['create', 'list', 'total_area', 'total_perimeter', 'clear']:
                calculator.process_shape_command(command, *args)
            elif command in ['prime', 'fibonacci', 'prime_from']:
                calculator.process_sequence_command(command, *args)
            else:
                print("Неизвестная команда. Используйте 'demo' для просмотра возможностей.")

        except KeyboardInterrupt:
            print("\n\n Программа прервана пользователем.")
            break
        except Exception as e:
            print(f"Критическая ошибка: {e}")
        finally:
            print("-" * 40)

if __name__ == "__main__":
    main()

ГЕОМЕТРИЧЕСКИЙ КАЛЬКУЛЯТОР И ГЕНЕРАТОР ПОСЛЕДОВАТЕЛЬНОСТЕЙ

Доступные команды:
ФИГУРЫ: create <тип> <параметры>, list, total_area, total_perimeter, clear
ПОСЛЕДОВАТЕЛЬНОСТИ: prime [n], fibonacci [n], prime_from <start> <n>
ДЕМО: demo, ВЫХОД: exit

Введите команду: create circle 5
Создана фигура: Круг: площадь=78.54, периметр=31.42
Команда выполнена!
----------------------------------------

Доступные команды:
ФИГУРЫ: create <тип> <параметры>, list, total_area, total_perimeter, clear
ПОСЛЕДОВАТЕЛЬНОСТИ: prime [n], fibonacci [n], prime_from <start> <n>
ДЕМО: demo, ВЫХОД: exit

Введите команду: create rectangle 4 6
Создана фигура: Прямоугольник: площадь=24.00, периметр=20.00
Команда выполнена!
----------------------------------------

Доступные команды:
ФИГУРЫ: create <тип> <параметры>, list, total_area, total_perimeter, clear
ПОСЛЕДОВАТЕЛЬНОСТИ: prime [n], fibonacci [n], prime_from <start> <n>
ДЕМО: demo, ВЫХОД: exit

Введите команду: list
Коллекция фигур:
  1. Круг: площадь=78.54, перим